# 01 · Data Generation
Generates the synthetic AdCore-ML dataset: ~150k rows spanning users, content, campaigns, engagement and financial fields, with intentionally injected missing values, outliers, and duplicates so the rest of the pipeline has real data-quality work to do.

In [1]:
import sys
sys.path.append('../src')
import pandas as pd
from data_generator import generate_dataset
pd.set_option('display.max_columns', 40)

In [2]:
df = generate_dataset(n_rows=150_000, seed=42)
print(f'Shape: {df.shape}')
df.head()

Shape: (150600, 30)


,user_id,age,gender,country,city,device_type,subscription_type,content_id,content_genre,content_duration,content_rating,campaign_id,ad_type,ad_placement,advertiser_category,bid_amount,impressions,clicks,conversions,clicked,session_duration,pages_viewed,cpm,cpc,revenue,cost,timestamp,hour,day_of_week,month
0,278081,60.0,M,US,Chicago,desktop,basic,8883,drama,27.6,PG-13,5893,interstitial,pre_roll,cpg,3.73,1,0,0,0,20.7,2,9.50,1.37,0.00,0.01,2024-04-18 04:00:00,4,3,4
1,990984,44.0,M,IN,Hyderabad,desktop,premium,6863,comedy,51.2,G,5467,video,sidebar,tech,2.52,1,0,0,0,25.3,6,12.40,1.24,0.00,0.01,2025-04-22 22:00:00,22,1,4
2,137269,27.0,F,BR,Sao Paulo,desktop,family,2281,news,37.7,PG-13,5896,native,mid_roll,retail,5.80,7,2,1,1,34.8,2,9.19,1.02,14.45,0.37,2025-02-17 11:00:00,11,0,2
3,366667,50.0,F,US,New York,mobile,basic,3348,news,27.9,PG,5877,video,in_feed,tech,2.35,5,0,0,0,1.6,3,16.96,1.44,0.03,0.08,2024-10-23 23:00:00,23,2,10
4,609676,74.0,M,MX,Mexico City,desktop,premium,2808,drama,23.2,PG,5174,native,sidebar,travel,3.49,3,0,0,0,20.6,2,7.69,0.97,0.01,0.02,2024-10-20 19:00:00,19,6,10


## Schema

In [3]:
df.dtypes

user_id                         int64
age                           float64
gender                            str
country                           str
city                              str
device_type                       str
subscription_type                 str
content_id                      int64
content_genre                     str
content_duration              float64
content_rating                    str
campaign_id                     int64
ad_type                           str
ad_placement                      str
advertiser_category               str
bid_amount                    float64
impressions                     int64
clicks                          int64
conversions                     int64
clicked                         int64
session_duration              float64
pages_viewed                    int64
cpm                           float64
cpc                           float64
revenue                       float64
cost                          float64
timestamp   

## A quick look at the intentional data-quality issues

In [4]:
print('Missing values per column (top 10):')
print(df.isna().sum().sort_values(ascending=False).head(10))
print()
print('Rows where clicks > impressions (tracking bug):',
      (df['clicks'] > df['impressions']).sum())
print('Rows with negative cost (billing glitch):', (df['cost'] < 0).sum())
print('Duplicate rows:', df.duplicated().sum())

Missing values per column (top 10):
session_duration     4513
content_rating       3016
age                  3013
city                 3008
gender               2258
bid_amount           1511
device_type             0
subscription_type       0
content_genre           0
content_id              0
dtype: int64

Rows where clicks > impressions (tracking bug): 225
Rows with negative cost (billing glitch): 149


Duplicate rows: 600


## Save to disk
This is the same file used by the rest of the pipeline (`data/adcore_raw.csv`).

In [5]:
df.to_csv('../data/adcore_raw.csv', index=False)
print('Saved', len(df), 'rows to data/adcore_raw.csv')

Saved 150600 rows to data/adcore_raw.csv
